# Importação de Bibliotecas
Carregamento das bibliotecas necessárias para a análise.

In [8]:
import pandas as pd
import numpy as np

# Carregamento dos Dados
Carregamento dos conjuntos de dados da PAM, PPM e PIB.

In [9]:
# Lista dos 5 arquivos da PAM
arquivos_pam = [
    "../dados/dados_originais/t5457_pam_area_colhida_2003_2024_ce_br.csv",
    "../dados/dados_originais/t5457_pam_area_plantada_2003_2024_ce_br.csv",
    "../dados/dados_originais/t5457_pam_quantidade_produzida_2003_2024_ce_br.csv",
    "../dados/dados_originais/t5457_pam_rendimento_medio_2003_2024_ce_br.csv",
    "../dados/dados_originais/t5457_pam_valor_producao_2003_2024_ce_br.csv"
]

# Carregando e concatenando dos aquivos da PAM
dfs_pam = []
for arquivo in arquivos_pam:
    df = pd.read_csv(arquivo, sep=";", dtype={"territorio_codigo": "string", "variavel_codigo": "string", "produto_codigo": "string"})
    dfs_pam.append(df)

df_pam = pd.concat(dfs_pam, ignore_index=True)

# Carregando PPM e PIB
df_ppm = pd.read_csv("../dados/dados_originais/t3939_ppm_efetivo_rebanhos_2003_2024_ce_br.csv", sep=";", dtype={"territorio_codigo": "string", "variavel_codigo": "string", "tipo_rebanho_codigo": "string"})
df_pib = pd.read_csv("../dados/dados_originais/t5938_pib_agropecuaria_2003_2023_ce_br.csv", sep=";", dtype={"territorio_codigo": "string", "variavel_codigo": "string"})

# Definição das Chaves Únicas
Estabelecendo as chaves únicas conforme definido no README do projeto.

In [10]:
chaves_pam = ['nivel_territorial_codigo', 'territorio_codigo', 'ano_codigo', 'variavel_codigo', 'produto_codigo']
chaves_ppm = ['nivel_territorial_codigo', 'territorio_codigo', 'ano_codigo', 'variavel_codigo', 'tipo_rebanho_codigo']
chaves_pib = ['nivel_territorial_codigo', 'territorio_codigo', 'ano_codigo', 'variavel_codigo']

# Validação de Qualidade 1: Teste de Duplicatas
Agrupa as tasks: **Verificar duplicatas PAM**, **Verificar duplicatas PPM** e **Verificar duplicatas PIB**.

Se o retorno for `False`, a base está perfeitamente livre de duplicatas em suas respectivas chaves.

In [11]:
print("--- DIAGNÓSTICO DE DUPLICATAS ---")
print(f"Duplicatas na PAM (Consolidada): {df_pam.duplicated(subset=chaves_pam).any()}")
print(f"Duplicatas na PPM (Efetivo): {df_ppm.duplicated(subset=chaves_ppm).any()}")
print(f"Duplicatas no PIB: {df_pib.duplicated(subset=chaves_pib).any()}")

--- DIAGNÓSTICO DE DUPLICATAS ---
Duplicatas na PAM (Consolidada): False
Duplicatas na PPM (Efetivo): False
Duplicatas no PIB: False


# Validação de Qualidade 2: Mapeamento de Símbolos Especiais e Nulos
Aqui realizamos a task de **Quantificar ausências** (valores nulos literais) e mapeamos simbolos especiais existentes nas tabelas.

In [12]:
def mapear_simbolos_e_nulos(df, nome_tabela):
    simbolos = df[~df['valor'].astype(str).str.match(r'^-?\d+\.?\d*$', na=False)]['valor'].unique()
    nulos = df['valor'].isnull().sum()
    print(f"\n--- QUALIDADE DA TABELA {nome_tabela} ---")
    print(f"Símbolos encontrados: {simbolos}")
    print(f"Valores vazios literais (NaN): {nulos}")

mapear_simbolos_e_nulos(df_pam, 'PAM (Consolidada)')
mapear_simbolos_e_nulos(df_ppm, 'PPM (Efetivo)')
mapear_simbolos_e_nulos(df_pib, 'PIB')


--- QUALIDADE DA TABELA PAM (Consolidada) ---
Símbolos encontrados: <StringArray>
['-']
Length: 1, dtype: str
Valores vazios literais (NaN): 0

--- QUALIDADE DA TABELA PPM (Efetivo) ---
Símbolos encontrados: <StringArray>
['-']
Length: 1, dtype: str
Valores vazios literais (NaN): 0

--- QUALIDADE DA TABELA PIB ---
Símbolos encontrados: <StringArray>
['...']
Length: 1, dtype: str
Valores vazios literais (NaN): 0


# Validação de Qualidade 3: Identificação de Símbolos Específicos
Agrupa as tasks: **Identificar -**, **Identificar 0**, **Identificar X**, **Identificar ..** e **Identificar ...**.

Aqui contamos a frequência exata destes símbolos para posterior tratamento.

In [13]:
print("\n--- CONTAGEM EXATA DE SÍMBOLOS ---")
print("PAM (Consolidada):")
print(df_pam['valor'].astype(str).str.strip().value_counts()[lambda x: x.index.isin(['-', '...', 'X', '..', '0'])])

print("\nPPM (Efetivo):")
print(df_ppm['valor'].astype(str).str.strip().value_counts()[lambda x: x.index.isin(['-', '...', 'X', '..', '0'])])

print("\nPIB:")
print(df_pib['valor'].astype(str).str.strip().value_counts()[lambda x: x.index.isin(['-', '...', 'X', '..', '0'])])


--- CONTAGEM EXATA DE SÍMBOLOS ---
PAM (Consolidada):
valor
-    31467
0        6
Name: count, dtype: int64

PPM (Efetivo):
valor
-    5
Name: count, dtype: int64

PIB:
valor
...    1116
Name: count, dtype: int64
